# Academic Success: EDA which makes sense

This notebook is an exploratory data analysis for the academic success competition. It shows:
- an analysis of the data
- how to deal with the categorical features
- how to cross-validate
- several strong models
- how to ensemble the predictions

References: 
- Competition: [Classification with an Academic Success Dataset](https://www.kaggle.com/competitions/playground-series-s4e6/)
- Data description: [UC Irvine repository](https://archive.ics.uci.edu/dataset/697/predict+students+dropout+and+academic+success)
- Introductory paper: [Early prediction of student's performance in higher education: a case study](https://www.mdpi.com/2306-5729/7/11/146)

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from matplotlib.ticker import MaxNLocator
import seaborn as sns
from colorama import Fore, Style
import xgboost
import lightgbm
import catboost
import os
import datetime
import pickle
import warnings
from scipy.stats import mannwhitneyu
import scipy.special

from sklearn.model_selection import StratifiedKFold, cross_val_score, cross_val_predict 
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import StandardScaler, FunctionTransformer, SplineTransformer
from sklearn.preprocessing import OneHotEncoder, LabelEncoder, label_binarize
from sklearn.pipeline import make_pipeline, Pipeline
from sklearn.linear_model import LogisticRegression, LogisticRegressionCV, Ridge, RidgeCV
from sklearn.svm import LinearSVC
from sklearn.tree import DecisionTreeClassifier, plot_tree
from sklearn.ensemble import ExtraTreesClassifier, RandomForestClassifier, BaggingClassifier, HistGradientBoostingClassifier
from sklearn.inspection import permutation_importance
from sklearn.metrics import log_loss, accuracy_score
from sklearn.calibration import CalibrationDisplay


In [ ]:
# Configuration
# Produce a submission file (you can set this to false if you only
# want to see the cross-validation results)
COMPUTE_TEST_PRED = True

# You can decide whether to use the original dataset for training
# Recommendation: yes, use it!
USE_ORIGINAL = True

# Containers for results
oof, test_pred, fold_scores = {}, {}, {}

# About the dataset

In the dataset of this competition, every sample corresponds to a student at a Portuguese university. There are 4424 students in the original dataset, and Kaggle has generated more than 100000 synthetic samples based on the original dataset.

For every student, we get demographic data, macroeconomic data and the performance during the first two semesters of the course. The competition is about predicting the state of the student after three or four years of study: Either graduated, still enrolled, or dropped out.

# Reading the data

We read the data and see that the training dataset has about 76000 samples and 36 features. The original dataset, from which the synthetic data was generated, is much smaller (4424 samples).

Insight:
1. We'll need algorithms which can handle that much data (i.e., no kernel matrices).
2. Maybe for that much data we'll not always do a five-fold cross-validation. If we do only a simple train–test split, we can save some time.


In [ ]:
train = pd.read_csv('/kaggle/input/playground-series-s4e6/train.csv', index_col='id')
test = pd.read_csv('/kaggle/input/playground-series-s4e6/test.csv', index_col='id')
original = pd.read_csv('/kaggle/input/predict-students-dropout-and-academic-success/data.csv')

initial_features = list(test.columns)
train

# Target distribution

There are three target classes: we have a multiclass classification task. The classes are not severely unbalanced.

We encode the targets with a `LabelEncoder`.

In [ ]:
temp = train.Target.value_counts()
temp

plt.figure(figsize=(6, 3))
plt.pie(temp, labels=temp.index, autopct="%.0f%%")
plt.title('Target distribution')
plt.show()

In [ ]:
label_encoder = LabelEncoder()
targets = label_encoder.fit_transform(train.Target)
original_targets = label_encoder.transform(original.Target)

# Feature distributions

According to the [variables table](https://archive.ics.uci.edu/dataset/697/predict+students+dropout+and+academic+success), some features are categorical. Note that we cannot distinguish categorical features from integer features by looking at the data. We can only distinguish them by looking at the documentation!

We encode the categorical features with a categorical dtype (and we might one-hot encode them later depending on the algorithm we want to apply) because good models perform better if they are given the information which features are categorical:


In [ ]:
cat_features = ['Marital status', 'Application mode', 'Course',
                'Previous qualification', 'Nacionality', "Mother's qualification", 
                "Father's qualification", "Mother's occupation",
                "Father's occupation"]
for feature in cat_features:
    dtype = pd.CategoricalDtype(categories=list(set(train[feature]) | set(test[feature]) | set(original[feature])), ordered=False)
    for df in [train, test, original]:
        df[feature] = df[feature].astype(dtype)

We can now plot the feature distributions:
- Integer features (e.g., application order) are plotted as blue bar charts.
- Categorical features (e.g., marital status) are plotted as black bar charts.
- Float features (e.g., admission grade) are plotted as green histograms.

Some features have really uneven distributions. For instance, there are 18 nationalities, but 99 % of the students are Portuguese.

Most of the float features have only few unique values, for instance, GDP has only 11 unique values. By the way, although the feature is called GDP, it has negative values. Maybe it represents a GDP growth rate.

In [ ]:
_, axs = plt.subplots(9, 4, figsize=(12, 20))
for col, ax in zip(initial_features, axs.ravel()):
    if train[col].dtype == float:
        ax.hist(train[col], bins=300, density=True, color='g')
    elif train[col].dtype == 'category':
        vc = train[col].cat.codes.value_counts() / len(train)
        ax.bar(vc.index, vc, color='k')
        ax.yaxis.set_major_formatter('{x:.0%}')
        ax.set_xticks([])
    else: # integer
        vc = train[col].value_counts() / len(train)
        ax.bar(vc.index, vc, color='b')
        ax.xaxis.set_major_locator(MaxNLocator(integer=True))
        ax.yaxis.set_major_formatter('{x:.0%}')
    ax.set_title(col, fontsize=10)
plt.tight_layout()
plt.show()

# Feature correlations

DISCLAIMER: It is very questionable whether one should compute correlations for categorical features, but we do it anyway.

In the heatmap, all correlations are multiplied by 10 for better readability.

In [ ]:
corr_features = initial_features
cc = np.corrcoef(train[corr_features], rowvar=False)
plt.figure(figsize=(11, 11))
sns.heatmap(cc*10, center=0, cmap='coolwarm', annot=True, fmt='.0f',
            xticklabels=corr_features, yticklabels=corr_features)
plt.title('Correlation matrix')
plt.show()

You can detect and explain several correlations, for instance:
- All the statistics about "curricular units" are correlated.
- Nationality is correlated to the binary 'international' feature.
- Mother's and father's occupation are highly correlated.
- A bunch of 'curricular units' features are correlated.
- Marital status is correlated with age.

# Comparing train, test and the original data

A standard question in Kaggle playground competitions is whether the three datasets train, test and original have the same distributions. The question can be answered informally by histograms of a single feature:
- In a comparison between training data and original data, the leftmost bar of the histogram shows that they have different unconditional distributions (the conditional target distributions may still be the same, though).
- The distributions of train and test are almost the same.

In [ ]:
plt.figure(figsize=(6, 2))
plt.hist(train['Curricular units 2nd sem (grade)'],
         bins=np.linspace(0, 19, 31),
         density=True,
         color='m',
         label='train')
plt.hist(original['Curricular units 2nd sem (grade)'],
         bins=np.linspace(0, 19, 31),
         density=True,
         alpha=0.8,
         color='c',
         label='original')
plt.xlabel('Curricular units 2nd sem (grade)')
plt.ylabel('density')
plt.title('Train and original are different')
plt.legend()
plt.show()

plt.figure(figsize=(6, 2))
plt.hist(train['Curricular units 2nd sem (grade)'],
         bins=np.linspace(0, 19, 31),
         density=True,
         color='m',
         label='train')
plt.hist(test['Curricular units 2nd sem (grade)'],
         bins=np.linspace(0, 19, 31),
         density=True,
         alpha=0.8,
         color='g',
         label='test')
plt.xlabel('Curricular units 2nd sem (grade)')
plt.ylabel('density')
plt.title('Train and test are slightly different')
plt.legend()
plt.show()

More formally, we can compare the distributions with a [Mann–Whitney U test](https://en.wikipedia.org/wiki/Mann%E2%80%93Whitney_U_test). The low pvalue for the original dataset indicates that the distributions are different. The pvalue for the test dataset is 0.06, which means we shouldn't reject the null hypothesis that the train and test distributions are the same.

In [ ]:
print('Mann–Whitney U test comparing train and original datasets')
print(mannwhitneyu(train['Curricular units 2nd sem (grade)'],
                   original['Curricular units 2nd sem (grade)']))
print('\nMann–Whitney U test comparing train and test datasets')
print(mannwhitneyu(train['Curricular units 2nd sem (grade)'],
                   test['Curricular units 2nd sem (grade)']))


**Important:** Although the distributions of train and original differ significantly, adding the original dataset to the training dataset improves the score. And only the score matters — forget these histograms and Mann–Whitney U tests if they don't help improve your models' scores!

# The two most important features

A decision tree shows us the two most important predictive features:
- If `Curricular units 2nd sem (approved)` is 5 or more, the student will graduate at the end of the three-year course.
- If `Curricular units 2nd sem (approved)` is 0 or 1, the student will drop out before the end of the three-year course.
- Otherwise, check the tuition fees. If they are up to date, the student will be enrolled, but not yet graduated; if they are not, the student will drop out.

See [here](https://www.kaggle.com/competitions/playground-series-s4e6/discussion/509073) for a more thorough discussion.

In [ ]:
dt = DecisionTreeClassifier(max_depth=3)
dt.fit(train[initial_features], train.Target);

plt.figure(figsize=(16, 6))
plot_tree(dt, feature_names=initial_features, class_names=label_encoder.classes_, fontsize=7, impurity=False, filled=True, ax=plt.gca())
plt.show()

# Cross-validation

Before analyzing the data more, I want to see what a few simple models do with the data. To ensure that our cross-validation results are consistent, we'll use the same function for cross-validating all models.

The function does a little more than just predicting the classes for the test data:
- It fits the model in a `StratifiedKFold` and saves the predicted out-of-fold probabilities. The probabilities will later be useful when we ensemble the models. They can always be converted into class predictions by `np.argmax()`.
- It adds the original data to the training data. (Don't modify the validation data!)
- It saves the cross-validation scores so that we can compare and rank the models at the end of the notebook.
- It refits the model to the full training dataset and saves the predicted probabilities for the test dataset.


In [ ]:
crossval_kf = StratifiedKFold(n_splits=5, shuffle=True, random_state=1)

def cross_validate(model, label, features=initial_features):
    """Compute out-of-fold and test predictions for a given model.
    
    Out-of-fold and test predictions are stored in the global variables
    oof and test_pred, respectively.
    """
    start_time = datetime.datetime.now()
    scores = []
    oof_preds = np.full((len(train), 3), np.nan, dtype=float)
    for fold, (idx_tr, idx_va) in enumerate(crossval_kf.split(train, targets)):
        X_tr = train.iloc[idx_tr][features]
        X_va = train.iloc[idx_va][features]
        y_tr = targets[idx_tr]
        y_va = targets[idx_va]
        
        if USE_ORIGINAL:
            X_tr = pd.concat([X_tr, original[features]], axis=0)
            y_tr = np.hstack([y_tr, original_targets])
        
        model.fit(X_tr, y_tr)
        y_pred = model.predict_proba(X_va)
        
        score = accuracy_score(y_va, np.argmax(y_pred, axis=1))
        print(f"# Fold {fold}: accuracy={score:.5f}")
        scores.append(score)
        oof_preds[idx_va] = y_pred
            
    elapsed_time = datetime.datetime.now() - start_time
    accuracy = accuracy_score(targets, np.argmax(oof_preds, axis=1))
    logloss = log_loss(targets, oof_preds)   
    print(f"{Fore.GREEN}# Overall: accuracy={accuracy:.5f}"
          f" logloss={logloss:.5f} {label}"
          f"   {int(np.round(elapsed_time.total_seconds() / 60))} min{Style.RESET_ALL}")
    oof[label] = oof_preds
    fold_scores[label] = scores
    
    if COMPUTE_TEST_PRED:
        X_tr = train[features]
        y_tr = targets

        if USE_ORIGINAL:
            X_tr = pd.concat([X_tr, original[features]], axis=0)
            y_tr = np.hstack([y_tr, original_targets])
        
        model.fit(X_tr, y_tr)
        y_pred = model.predict_proba(test[features])
        test_pred[label] = y_pred

# Models

We start with ExtraTrees and random forests. Note that the model with the best (lowest) log loss is not necessarily the model with the best (highest) accuracy. It matters what metric you use to train and select models!

In [ ]:
et_params = {'min_samples_leaf': 1, 'max_features': 0.4228007626245592, 'min_impurity_decrease': 4.004184188705882e-05, 'n_estimators': 100, 'criterion': 'log_loss'}
model = ExtraTreesClassifier(**et_params)
cross_validate(model, 'ExtraTrees 1')
# Overall: accuracy=0.82566 logloss=0.45212 ExtraTrees   2 min

In [ ]:
et_params = {'min_samples_leaf': 4, 'max_features': 0.42393862999688375, 'min_impurity_decrease': 5.314080269075048e-06, 'n_estimators': 100, 'criterion': 'log_loss'}
model = ExtraTreesClassifier(**et_params)
cross_validate(model, 'ExtraTrees 2')
# Overall: accuracy=0.82600 logloss=0.46060 ExtraTrees 2   2 min

In [ ]:
model = RandomForestClassifier(n_estimators=400, min_impurity_decrease=1e-6)
cross_validate(model, 'Random forest 1')
# Overall: accuracy=0.82824 logloss=0.45231 Random forest 1   6 min

In [ ]:
rf_params = {'min_samples_leaf': 2, 'max_features': 0.6678943947450321, 'min_impurity_decrease': 0.00014125090018120894, 'max_samples': 0.8660526124724528, 'n_estimators': 400, 'criterion': 'log_loss'}
model = RandomForestClassifier(**rf_params)
cross_validate(model, 'Random forest 2')
# Overall: accuracy=0.82625 logloss=0.44824 Random forest 2   12 min

Next, we compare XGBoost with default parameters against tuned XGBoost models. The tuned models perform much better, but they take longer to train (they create 768 or more trees rather than only 100). The hyperparameters were found by Optuna.

In [ ]:
model = xgboost.XGBClassifier(enable_categorical=True)
cross_validate(model, 'XGBoost untuned')
# Overall: accuracy=0.83150 logloss=0.43800 XGBoost untuned   0 min

In [ ]:
xgb_params = {'grow_policy': 'depthwise', 'tree_method': 'hist', 'enable_categorical': True, 'gamma': 0, 'n_estimators': 768, 'learning_rate': 0.026111403303690425, 'max_depth': 8, 'reg_lambda': 26.648168065161098, 'min_child_weight': 1.0626186255116183, 'subsample': 0.8580490989206254, 'colsample_bytree': 0.5125814118774029}
model = xgboost.XGBClassifier(**xgb_params)
cross_validate(model, 'XGBoost tuned 1')
# Overall: accuracy=0.83443 logloss=0.42862 XGBoost tuned 1   3 min

In [ ]:
xgb_params = {'grow_policy': 'depthwise', 'learning_rate': 0.06150883051411711, 'n_estimators': 822, 'max_depth': 5, 'reg_lambda': 12.695544291635334, 'min_child_weight': 25.289808052903343, 'subsample': 0.9831949009517902, 'colsample_bytree': 0.2687930272243655, 'tree_method': 'hist', 'enable_categorical': True, 'gamma': 0} # 0.68179
model = xgboost.XGBClassifier(**xgb_params)
cross_validate(model, 'XGBoost tuned 2')
# Overall: accuracy=0.83433 logloss=0.42685 XGBoost tuned 2   2 min

In [ ]:
xgb_params = {'grow_policy': 'depthwise', 'learning_rate': 0.04104089631389812, 'n_estimators': 1311, 'max_depth': 5, 'reg_lambda': 29.548955808402486, 'min_child_weight': 17.58377776073493, 'subsample': 0.9141573846486278, 'colsample_bytree': 0.4000772723424121, 'tree_method': 'hist', 'enable_categorical': True, 'gamma': 0} # 0.68265
model = xgboost.XGBClassifier(**xgb_params)
cross_validate(model, 'XGBoost tuned 3')
# Overall: accuracy=0.83507 logloss=0.42728 XGBoost tuned 3   3 min

We do the same experiment with LightGBM: We cross-validate it once with default parameters and once with tuned parameters:

In [ ]:
model = lightgbm.LGBMClassifier(verbose=-1)
cross_validate(model, 'LightGBM untuned')
# Overall: accuracy=0.83282 logloss=0.43358 LightGBM untuned   1 min

In [ ]:
lgbm_params = {'boosting_type': 'gbdt', 'verbose': -1, 'n_estimators': 680, 'learning_rate': 0.03, 'colsample_bytree': 0.5, 'reg_lambda': 1.8, 'min_child_samples': 95, 'num_leaves': 56}
model = lightgbm.LGBMClassifier(**lgbm_params)
cross_validate(model, 'LightGBM tuned')
# Overall: accuracy=0.83515 logloss=0.42822 LightGBM tuned   4 min

For CatBoost and DART, we show only the tuned versions. If you want to learn more about DART, I recommend you read the paper [DART: Dropouts meet Multiple Additive Regression Trees](https://arxiv.org/pdf/1505.01866.pdf).

In [ ]:
cb_params = {'grow_policy': 'Lossguide', 'max_depth': 12, 'verbose': False, 'n_estimators': 924, 'learning_rate': 0.0177946948928925, 'l2_leaf_reg': 5.726268912079757, 'max_leaves': 260, 'min_child_samples': 74, 'colsample_bylevel': 0.5376669237469467, 'random_strength': 0.3825264049149068}
model = catboost.CatBoostClassifier(**cb_params, cat_features=np.array(initial_features)[train[initial_features].dtypes == 'category'])
cross_validate(model, 'CatBoost tuned')
# Overall: accuracy=0.83285 logloss=0.43389 CatBoost tuned   22 min

In [ ]:
dart_params = {'boosting_type': 'dart', 'learning_rate': 0.4927780573536302, 'n_estimators': 386, 'colsample_bytree': 0.3405129756925218, 'reg_lambda': 198.978101237303, 'min_child_samples': 14, 'num_leaves': 11, 'verbose': -1} # 0.67833
model = lightgbm.LGBMClassifier(**dart_params)
cross_validate(model, 'DART')
# Overall: accuracy=0.83448 logloss=0.42837 DART   7 min

If we want to implement logistic regression, we need to one-hot encode the categorical features. The `OneHotEncoder` will complain about categories which are so rare that they only occur in the validation data, but not in the training data. Do you remember that there are very few non-Portuguese students in the dataset?

Logistic regression can profit a lot if we combine it with a `SplineTransformer`, `PowerTransformer`,  `PolynomialFeatures` or `Nystroem` transformer in a pipeline. In the current competition, logistic regression with the spline transformer overtakes some of the tree ensembles.

In [ ]:
# Logistic regression with one-hot and spline transformation
model = make_pipeline(ColumnTransformer([('ohe',
                                          OneHotEncoder(drop='first', sparse_output=False, handle_unknown='ignore'),
                                          train.select_dtypes('category').columns),
                                        ],
                                        remainder=SplineTransformer()),
                      StandardScaler(),
                      LogisticRegression(max_iter=1500))
cross_validate(model, 'LogisticRegression')
# Overall: accuracy=0.82669 logloss=0.45293 LogisticRegression   5 min

# Ensembles

We try three methods for ensembling the classifiers:
1. The first method (hard voting) takes a majority vote. Whatever class gets the most votes will be the final prediction. It looks like the majority vote doesn't improve the accuracy of the predictions.
2. The second method is ridge regression based on the predicted probabilities. It often gives the best accuracy (but a bad log loss).
3. The third method is logistic regression based on the predicted probabilities. In all my experiments, the logistic regression ensemble improved the log loss,  but the effect of ensembling on accuracy was less evident.

See [this discussion](https://www.kaggle.com/competitions/playground-series-s4e6/discussion/509353) for alternative ensembling methods.

In [ ]:
# Hard majority vote
X = np.column_stack([np.argmax(oof[label], axis=1) for label in oof.keys() if 'Ensemble' not in label])
print(f"{Fore.GREEN}# Hard voting ensemble accuracy={accuracy_score(targets, scipy.stats.mode(X, axis=1)[0]):.5f}{Style.RESET_ALL}")
# Hard voting ensemble accuracy=0.83437

In [ ]:
%%time
# Stacking with ridge regression
X = np.hstack([oof[label] for label in oof.keys() if 'Ensemble' not in label])
X_te = np.hstack([test_pred[label] for label in oof.keys() if 'Ensemble' not in label])
model = Ridge()
oof['Ensemble (ridge)'] = cross_val_predict(model,
                                            X, label_binarize(targets, classes=[0, 1, 2]),
                                            cv=crossval_kf.split(X, targets), method='predict')
oof['Ensemble (ridge)'] = oof['Ensemble (ridge)'].clip(0, 1)
print(f"{Fore.GREEN}# Stacking ensemble (ridge)"
      f" accuracy={accuracy_score(targets, np.argmax(oof['Ensemble (ridge)'], axis=1)):.5f}"
      f" logloss={log_loss(targets, oof['Ensemble (ridge)']):.5f}{Style.RESET_ALL}")
if COMPUTE_TEST_PRED:
    model.fit(X, label_binarize(targets, classes=[0, 1, 2]))
#     with np.printoptions(linewidth=150, edgeitems=20, precision=2, suppress=True, sign=' '):
#         print(model.coef_.T)
#         print(model.intercept_)
    test_pred['Ensemble (ridge)'] = model.predict(X_te)
# Stacking ensemble (ridge) accuracy=0.83545 logloss=0.43528

In [ ]:
%%time
# Stacking with logistic regression
X = np.hstack([oof[label] for label in oof.keys() if 'Ensemble' not in label]).clip(1e-15, 1-1e-15)
X_te = np.hstack([test_pred[label] for label in oof.keys() if 'Ensemble' not in label]).clip(1e-15, 1-1e-15)
model = make_pipeline(FunctionTransformer(np.log), LogisticRegression(max_iter=500))
oof['Ensemble (logistic)'] = cross_val_predict(model,
                                               X, targets,
                                               cv=crossval_kf, method='predict_proba')
print(f"{Fore.GREEN}# Stacking ensemble (logistic)"
      f" accuracy={accuracy_score(targets, np.argmax(oof['Ensemble (logistic)'], axis=1)):.5f}"
      f" logloss={log_loss(targets, oof['Ensemble (logistic)']):.5f}{Style.RESET_ALL}")
if COMPUTE_TEST_PRED:
    model.fit(X, targets)
    test_pred['Ensemble (logistic)'] = model.predict_proba(X_te)
# Stacking ensemble (logistic) accuracy=0.83481 logloss=0.42635

The calibration of the stacked predictions looks ok:

In [ ]:
# Stacking ensemble calibration display
plt.figure(figsize=(12, 3))
for target in range(3):
    plt.subplot(1, 3, target+1)
    CalibrationDisplay.from_predictions(targets == target,
                                        oof['Ensemble (logistic)'][:, target],
                                        n_bins=50, 
                                        strategy='quantile',
                                        name='Ensemble (logistic)',
                                        ax=plt.gca())
    plt.xlabel(f'Mean predicted probability for {label_encoder.classes_[target]}')
    plt.ylabel(f'Fraction of positives for {label_encoder.classes_[target]}')
plt.suptitle('Calibration')
plt.show()

# Evaluation

Now look at the ranking and decide yourself what models you're going to refine and which ones to drop. Notice the importance of hyperparameter tuning!

In [ ]:
result_list = []
for label in oof.keys():
    score = accuracy_score(targets, np.argmax(oof[label], axis=1))
    result_list.append((label, score))
result_df = pd.DataFrame(result_list, columns=['label', 'score'])
result_df.sort_values('score', inplace=True, ascending=False)

plt.figure(figsize=(12, len(result_df) * 0.4 + 0.4))
bars = plt.barh(np.arange(len(result_df)), result_df.score, color='lightgreen')
plt.gca().bar_label(bars, fmt='%.5f')
plt.yticks(np.arange(len(result_df)), result_df.label)
plt.gca().invert_yaxis()
plt.xlim(0.82, 0.84)
plt.xlabel(f'accuracy (higher is better)')
plt.show()

# Fold–fold and CV–LB correspondence

During Kaggle competitions, people often discuss how well cross-validation scores correspond to leaderboard scores. We can simulate this correspondence by matching two folds of our cross-validation. The diagrams show that the correspondence between folds is not too bad.

The correspondence between cv score and private leaderboard will be even better because they have higher numbers of samples:
- Single fold = 15304 samples
- Cross-validation = 76518 samples
- Public test = 10202 samples
- Private test = 40810 samples

In [ ]:
fold_score_df = pd.DataFrame(fold_scores, index=range(crossval_kf.n_splits)).T
_, axs = plt.subplots(1, 2, sharex=True, sharey=True, figsize=(10, 4))

sns.regplot(x=fold_score_df[0], y=fold_score_df[1], ax=axs[0], color='m')
axs[0].set_xlabel('Accuracy of fold 0')
axs[0].set_ylabel('Accuracy of fold 1')
axs[0].set_aspect('equal')

sns.regplot(x=fold_score_df[2], y=fold_score_df[3], ax=axs[1], color='m')
axs[1].set_xlabel('Accuracy of fold 2')
axs[1].set_ylabel('Accuracy of fold 3')
axs[1].set_aspect('equal')

plt.suptitle('Correspondence between fold scores')
plt.show()

# Submission

We create submission files for the model with the highest accuracy and for the two stacking ensembles.

In [ ]:
if COMPUTE_TEST_PRED:
    print(result_df.label.iloc[0])
    pred = np.argmax(test_pred[result_df.label.iloc[0]], axis=1)
    pred = label_encoder.inverse_transform(pred)
    sub = pd.Series(pred, index=test.index, name='Target')
    filename = 'submission.csv'
    sub.to_csv(filename)
    os.system(f"head {filename}")
    
    print('\nEnsemble (ridge)')
    pred = np.argmax(test_pred['Ensemble (ridge)'], axis=1)
    pred = label_encoder.inverse_transform(pred)
    sub = pd.Series(pred, index=test.index, name='Target')
    filename = 'submission_ensemble_ridge.csv'
    sub.to_csv(filename)
    os.system(f"head {filename}")
    
    print('\nEnsemble (logistic)')
    pred = np.argmax(test_pred['Ensemble (logistic)'], axis=1)
    pred = label_encoder.inverse_transform(pred)
    sub = pd.Series(pred, index=test.index, name='Target')
    filename = 'submission_ensemble_logistic.csv'
    sub.to_csv(filename)
    os.system(f"head {filename}")
    
    

# Conclusion

We have just started analyzing the data.

There is much more to do:
- Eliminate useless features
- Create additional features
- Implement additional models (and don't forget to tune them)
- Find a better ensembling method